# Task 3 — Multimodal Temporal Hypothesis Generation (Google Colab / Jupyter)

Этот блокнот является **продолжением Task 1 и Task 2** и предназначен для рабочего запуска третьей задачи:

- приём входа из **query**, **списка DOI / URL / identifier**, **командного списка** или **Task 1 YAML**;
- построение **multimodal temporal graph** и **ranked hypotheses**;
- использование **локальной VL модели из Hugging Face / Transformers** либо **g4f API**;
- сборка **офлайн A/B формы** для эксперта;
- сборка **архива артефактов для эксперта**;
- сохранение итогов в `runs/task3_hypotheses/...`.

> Блокнот хранится как обычный Jupyter notebook в формате `nbformat` JSON, поэтому его можно открывать и в Colab, и в Jupyter совместимых средах. citeturn686087search0turn686087search3

# Пошаговый туториал

## Базовый сценарий
1. Запустите ячейку **«Установка и импорт»**.
2. В форме ниже выберите один из входов:
   - `Query` — свободный поисковый запрос;
   - `Task 1 YAML` — если уже есть экспертная траектория;
   - `Commands` — если хотите управлять запуском списком команд;
   - `Processed papers ZIP` — если хотите запустить Task 3 офлайн на уже подготовленных артефактах.
3. Выберите routing для моделей:
   - **g4f** для API-моделей;
   - **Transformers / HF local VLM** для локальной vision-language модели.
4. Запустите отдельную нижнюю ячейку **«Запуск Task 3»**.
5. После окончания получите:
   - `task3_manifest.json`
   - `hypotheses_ranked.json` и `hypotheses_ranked.md`
   - `task3_hypothesis_review_offline_ab.html`
   - `expert_hypothesis_artifacts_bundle.zip`

## Формат commands
Можно вставить текст или загрузить `.txt / .yaml / .json`, например:

```text
query: temporal knowledge graph multimodal hypothesis generation
identifier: 10.1038/s41586-020-2649-2
identifier: https://doi.org/10.1126/science.abc1234
top_papers: 12
top_hypotheses: 8
edge_mode: auto
link_prediction_backend: auto
g4f_model: auto
vlm_backend: qwen2_vl
vlm_model_id: Qwen/Qwen2.5-VL-3B-Instruct
```

## Что делает A/B форма
Оффлайн HTML показывает **две версии гипотезы** в слепом порядке:
- **Task 3 full model** — полная гипотеза из pipeline;
- **Candidate template baseline** — более простой baseline на базе candidate signal.

Эксперт может выбрать предпочтительный вариант, оценить temporal reasoning, evidence grounding, testability, novelty и выгрузить ZIP с review.

In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
import gc, io, json, os, sys, tempfile, subprocess, zipfile, shutil
from pathlib import Path

os.environ.setdefault('G4F_ASYNC_ENABLED', '1')
os.environ.setdefault('G4F_ASYNC_MAX_CONCURRENCY', '3')
os.environ.setdefault('G4F_ASYNC_RETRIES', '3')
os.environ.setdefault('G4F_ASYNC_MAX_MODELS_PER_REQUEST', '3')
os.environ.setdefault('LLM_REQUEST_TIMEOUT_SECONDS', '25')
os.environ.setdefault('VLM_REQUEST_TIMEOUT_SECONDS', '45')

if os.environ.get('TASK3_NOTEBOOK_SMOKE') == '1':
    os.environ.setdefault('LLM_PROVIDER', 'mock')
    os.environ.setdefault('LLM_MODEL', 'mock')
    os.environ.setdefault('EMBED_PROVIDER', 'hash')
    os.environ.setdefault('MM_EMBED_BACKEND', 'none')
    os.environ.setdefault('VLM_BACKEND', 'none')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
    os.environ.setdefault('HF_DATASETS_OFFLINE', '1')



def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def run_optional(cmd, cwd=None, label=None):
    try:
        run(cmd, cwd=cwd)
        return True
    except Exception as e:
        label = label or ' '.join(map(str, cmd))
        print(f'[warn] optional step failed: {label}: {type(e).__name__}: {e}')
        return False


def _materialize_local_repo_archive() -> Path | None:
    search_roots = [Path('/mnt/data'), Path('/content'), Path.cwd(), Path.cwd().parent]
    patterns = (
        'top-papers-graph-main-task3-module.zip',
        'top-papers-graph-main.zip',
        'top-papers-graph*.zip',
        '*top-papers-graph*.zip',
    )
    for base in search_roots:
        if not base.exists():
            continue
        for pattern in patterns:
            for archive in sorted(base.glob(pattern)):
                target = archive.with_suffix('')
                try:
                    candidate_root = target / 'top-papers-graph-main'
                    if (candidate_root / 'pyproject.toml').exists() and (candidate_root / 'src' / 'scireason').exists():
                        return candidate_root
                    if target.exists() and (target / 'pyproject.toml').exists() and (target / 'src' / 'scireason').exists():
                        return target
                    target.mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(archive, 'r') as zf:
                        zf.extractall(target)
                    if (candidate_root / 'pyproject.toml').exists() and (candidate_root / 'src' / 'scireason').exists():
                        return candidate_root
                    if (target / 'pyproject.toml').exists() and (target / 'src' / 'scireason').exists():
                        return target
                except Exception as e:
                    print(f'[warn] local repo archive skipped: {archive}: {type(e).__name__}: {e}')
    return None


def find_repo_root() -> Path | None:
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        env_path = Path(env_dir)
        if (env_path / 'pyproject.toml').exists() and (env_path / 'src' / 'scireason').exists():
            return env_path

    candidates = []
    repo_from_archive = _materialize_local_repo_archive()
    if repo_from_archive is not None:
        candidates.append(repo_from_archive)
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data')])
    search_bases = [Path('/content'), Path('/mnt/data'), cwd, cwd.parent]
    patterns = ('top-papers-graph-main*', 'top-papers-graph*')
    for base in search_bases:
        if not base.exists():
            continue
        for pattern in patterns:
            candidates.extend(base.glob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else Path.cwd() / 'top-papers-graph'
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

SRC_DIR = REPO_DIR / 'src'
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Не найден каталог src в репозитории: {REPO_DIR}')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

run_optional([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets', 'pyyaml', 'requests', 'unidecode', 'nbformat'])
_task3_editable_install = [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[task3]']
if os.environ.get('TASK3_NOTEBOOK_SMOKE') == '1' or os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1' or os.environ.get('TASK3_NOTEBOOK_SKIP_OPTIONAL_DEPS') == '1':
    _task3_editable_install = [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps']
run_optional(_task3_editable_install, cwd=REPO_DIR, label='pip install editable task3 notebook runtime')

import yaml
import ipywidgets as W
from IPython.display import HTML, Markdown, display, clear_output
from unidecode import unidecode

from scireason.config import settings
from scireason.llm import list_available_g4f_models
from scireason.pipeline.task3_hypothesis_generation import prepare_task3_hypothesis_bundle
from scireason.task3_offline_review import (
    build_task3_expert_artifact_bundle,
    build_task3_offline_review_package,
)

TASK3_DEFAULT_G4F_MODEL = getattr(settings, 'task2_default_g4f_model', 'auto') or 'auto'
TASK3_DEFAULT_LOCAL_VLM_MODEL = getattr(settings, 'vlm_model_id', 'Qwen/Qwen2.5-VL-3B-Instruct') or 'Qwen/Qwen2.5-VL-3B-Instruct'
TASK3_LAST_RUN = None
TASK3_LAST_TASK_META = None
TASK3_LAST_ARTIFACTS = None

print('REPO_DIR =', REPO_DIR)
print('SRC_DIR  =', SRC_DIR)
print('task3 default g4f model =', TASK3_DEFAULT_G4F_MODEL)
print('task3 default local VLM model =', TASK3_DEFAULT_LOCAL_VLM_MODEL)

In [ ]:
# @title
# Вспомогательные функции (не нужно редактировать)
import ast
import hashlib
import json
import re
from datetime import datetime
from pathlib import Path


def _escape_html(value):
    return html_escape(str(value or ''))


def html_escape(text: str) -> str:
    return (
        str(text or '')
        .replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
        .replace('"', '&quot;')
        .replace("'", '&#39;')
    )


def _slugify(text: str) -> str:
    value = unidecode(str(text or '')).lower()
    value = re.sub(r'[^a-z0-9\s_-]+', '', value)
    value = re.sub(r'\s+', '_', value).strip('_')
    return value or 'expert'


def _materialize_upload(upload_value, out_dir: Path, default_name: str) -> Path | None:
    if not upload_value:
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        if not upload_value:
            return None
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        if content is None:
            return None
        path = out_dir / (name or default_name)
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        content = meta.get('content')
        if content is None:
            return None
        name = meta.get('name') or default_name
        path = out_dir / name
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    raise TypeError('Неподдерживаемый формат upload value')


def _read_upload_name_and_bytes(upload_value):
    if not upload_value:
        return None, None
    if isinstance(upload_value, dict) and upload_value:
        name, meta = next(iter(upload_value.items()))
        return name or 'upload.bin', meta.get('content')
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        return meta.get('name') or 'upload.bin', meta.get('content')
    return None, None


def _read_text_file(path: Path) -> str:
    return path.read_text(encoding='utf-8')


def _parse_bool(value, default=None):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on', 'да'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off', 'нет'}:
        return False
    return default


def _parse_identifier_blob(text: str):
    parts = []
    for raw_line in str(text or '').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        for token in re.split(r'[,;]', line):
            token = token.strip()
            if token:
                parts.append(token)
    return parts


def _parse_commands_payload(text: str) -> dict:
    payload = {
        'query': '',
        'identifiers': [],
        'domain_id': None,
        'top_papers': None,
        'top_hypotheses': None,
        'candidate_top_k': None,
        'search_limit': None,
        'edge_mode': None,
        'link_prediction_backend': None,
        'g4f_model': None,
        'local_model': None,
        'llm_provider': None,
        'llm_model': None,
        'vlm_backend': None,
        'vlm_model_id': None,
        'include_multimodal': None,
        'run_vlm': None,
        'processed_dir': None,
        'out_dir': None,
    }
    text = str(text or '').strip()
    if not text:
        return payload

    # JSON / YAML object first.
    for parser_name, parser in [('json', json.loads), ('yaml', yaml.safe_load)]:
        try:
            obj = parser(text)
            if isinstance(obj, dict):
                for key in list(payload.keys()):
                    if key == 'identifiers':
                        continue
                    if key in obj:
                        payload[key] = obj[key]
                if 'identifier' in obj:
                    payload['identifiers'].extend(_parse_identifier_blob(obj['identifier']))
                if 'identifiers' in obj:
                    if isinstance(obj['identifiers'], list):
                        payload['identifiers'].extend([str(x).strip() for x in obj['identifiers'] if str(x).strip()])
                    else:
                        payload['identifiers'].extend(_parse_identifier_blob(obj['identifiers']))
                return payload
            if isinstance(obj, list):
                for item in obj:
                    if isinstance(item, str):
                        payload['identifiers'].extend(_parse_identifier_blob(item))
                    elif isinstance(item, dict):
                        for key, value in item.items():
                            if key in {'identifier', 'id', 'doi', 'url'}:
                                payload['identifiers'].extend(_parse_identifier_blob(value))
                            elif key in payload:
                                payload[key] = value
                return payload
        except Exception:
            pass

    # Line based format.
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if ':' not in line:
            payload['identifiers'].append(line)
            continue
        key, value = line.split(':', 1)
        key = key.strip().lower()
        value = value.strip()
        if key in {'query', 'topic'}:
            payload['query'] = value
        elif key in {'identifier', 'id', 'doi', 'url'}:
            payload['identifiers'].extend(_parse_identifier_blob(value))
        elif key in {'identifiers', 'ids', 'doi_list', 'url_list'}:
            payload['identifiers'].extend(_parse_identifier_blob(value))
        elif key in {'domain', 'domain_id'}:
            payload['domain_id'] = value
        elif key in {'top_papers', 'top_hypotheses', 'candidate_top_k', 'search_limit'}:
            try:
                payload[key] = int(value)
            except Exception:
                pass
        elif key in {'edge_mode', 'link_prediction_backend', 'g4f_model', 'local_model', 'llm_provider', 'llm_model', 'vlm_backend', 'vlm_model_id', 'processed_dir', 'out_dir'}:
            payload[key] = value
        elif key in {'include_multimodal', 'run_vlm'}:
            payload[key] = _parse_bool(value, default=None)
    return payload


def _merged_commands_from_widgets(commands_text_value: str, commands_upload_value) -> dict:
    merged = _parse_commands_payload(commands_text_value)
    upload_name, upload_bytes = _read_upload_name_and_bytes(commands_upload_value)
    if upload_bytes:
        text = upload_bytes.decode('utf-8', errors='replace') if isinstance(upload_bytes, (bytes, bytearray)) else bytes(upload_bytes).decode('utf-8', errors='replace')
        uploaded = _parse_commands_payload(text)
        for key, value in uploaded.items():
            if key == 'identifiers':
                merged['identifiers'].extend([x for x in value if x])
            elif value not in (None, '', []):
                merged[key] = value
    deduped = []
    seen = set()
    for item in merged['identifiers']:
        norm = str(item).strip()
        if not norm or norm in seen:
            continue
        seen.add(norm)
        deduped.append(norm)
    merged['identifiers'] = deduped
    return merged


def _load_yaml_if_present(path: Path | None) -> dict:
    if path is None or not path.exists():
        return {}
    data = yaml.safe_load(path.read_text(encoding='utf-8')) or {}
    if isinstance(data, dict):
        return data
    raise ValueError('Ожидался YAML с top-level object')


def _discover_processed_dir(root: Path) -> Path | None:
    candidates = []
    for p in [root, *root.rglob('*')]:
        if not p.is_dir():
            continue
        has_meta = any(child.is_file() and child.name == 'meta.json' for child in p.glob('*/meta.json'))
        has_chunks = any(child.is_file() and child.name == 'chunks.jsonl' for child in p.glob('*/chunks.jsonl'))
        if has_meta or has_chunks:
            candidates.append(p)
    if candidates:
        candidates.sort(key=lambda p: len(p.parts))
        return candidates[0]
    return None


def _extract_processed_zip(upload_value, workdir: Path) -> Path | None:
    archive_path = _materialize_upload(upload_value, workdir, 'processed_papers.zip')
    if archive_path is None:
        return None
    extract_dir = workdir / 'processed_from_zip'
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(extract_dir)
    found = _discover_processed_dir(extract_dir)
    if found is None:
        raise FileNotFoundError('Не удалось найти processed_papers внутри ZIP-архива')
    return found


def _task3_task_meta_from_widgets(base_doc: dict | None = None):
    base_doc = dict(base_doc or {})
    expert = base_doc.get('expert') if isinstance(base_doc.get('expert'), dict) else {}
    last_name = str(expert_last.value or expert.get('last_name') or '').strip()
    first_name = str(expert_first.value or expert.get('first_name') or '').strip()
    patronymic = str(expert_pat.value or expert.get('patronymic') or '-').strip() or '-'
    full_name = ' '.join(x for x in [last_name, first_name, patronymic] if x).strip()
    latin_slug = _slugify(full_name)

    query_candidate = str(query.value or '').strip()
    submission_id = str(base_doc.get('submission_id') or '').strip()
    if not submission_id:
        seed_text = query_candidate or str(base_doc.get('topic') or 'task3')
        short_hash = hashlib.sha256(seed_text.encode('utf-8')).hexdigest()[:12]
        submission_id = f'{latin_slug or "expert"}__{short_hash}'

    topic_value = str(base_doc.get('topic') or query_candidate or '').strip()
    return {
        'topic': topic_value,
        'submission_id': submission_id,
        'cutoff_year': str(base_doc.get('cutoff_year') or '').strip(),
        'domain': str(base_doc.get('domain') or domain_id.value or '').strip(),
        'expert': {
            'last_name': last_name,
            'first_name': first_name,
            'patronymic': patronymic,
            'full_name': full_name,
            'latin_slug': latin_slug,
        },
    }


def _download_file_if_possible(path: Path, status_widget=None):
    try:
        from google.colab import files as colab_files
        colab_files.download(str(path))
        return True
    except Exception:
        if status_widget is not None:
            current = str(getattr(status_widget, 'value', '') or '')
            status_widget.value = current + f'<div><code>{_escape_html(str(path))}</code></div>'
        return False


def _top_hypotheses_summary(hypotheses_path: Path) -> str:
    try:
        rows = json.loads(hypotheses_path.read_text(encoding='utf-8'))
    except Exception:
        return 'Не удалось прочитать hypotheses_ranked.json'
    if not isinstance(rows, list) or not rows:
        return 'Гипотезы не были сгенерированы.'
    lines = []
    for row in rows[:5]:
        hyp = row.get('hypothesis') if isinstance(row.get('hypothesis'), dict) else {}
        title = str(hyp.get('title') or '(без названия)')
        lines.append(f"- **H-{int(row.get('rank') or 0):03d}** — {title}  ")
    return '\n'.join(lines)

In [ ]:
# @title
# Форма Task 3 (запустите ячейку и заполните поля)
from IPython.display import display

input_mode = W.Dropdown(
    options=[
        ('Task 1 YAML', 'yaml'),
        ('Query + identifiers', 'query'),
        ('Commands', 'commands'),
    ],
    value='query',
    description='Input mode',
)

trajectory_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Task1 YAML')
processed_upload = W.FileUpload(accept='.zip', multiple=False, description='Processed ZIP')
commands_upload = W.FileUpload(accept='.txt,.yaml,.yml,.json', multiple=False, description='Commands file')

query = W.Textarea(
    value='',
    description='Query',
    placeholder='temporal knowledge graph multimodal hypothesis generation',
    layout=W.Layout(width='100%', height='90px'),
)
identifiers_text = W.Textarea(
    value='',
    description='IDs',
    placeholder='DOI / URL / identifier — по одному на строку',
    layout=W.Layout(width='100%', height='110px'),
)
commands_text = W.Textarea(
    value='',
    description='Commands',
    placeholder='query: ...\nidentifier: ...\ntop_papers: 12\n...',
    layout=W.Layout(width='100%', height='140px'),
)

expert_last = W.Text(description='Фамилия', placeholder='Иванов', layout=W.Layout(width='100%'))
expert_first = W.Text(description='Имя', placeholder='Иван', layout=W.Layout(width='100%'))
expert_pat = W.Text(description='Отчество', placeholder='Иванович или -', value='-', layout=W.Layout(width='100%'))

domain_id = W.Text(value='science', description='Domain', layout=W.Layout(width='100%'))
out_dir = W.Text(value='runs/task3_hypotheses', description='Out dir', layout=W.Layout(width='100%'))
search_limit = W.IntSlider(value=25, min=5, max=100, step=5, description='Search limit', continuous_update=False)
top_papers = W.IntSlider(value=12, min=0, max=50, step=1, description='Top papers', continuous_update=False)
top_hypotheses = W.IntSlider(value=8, min=1, max=20, step=1, description='Top hyps', continuous_update=False)
candidate_top_k = W.IntSlider(value=16, min=4, max=50, step=1, description='Cand top-k', continuous_update=False)
include_multimodal = W.Checkbox(value=True, description='Include multimodal chunks')
run_vlm = W.Checkbox(value=True, description='Run VLM on pages / images / tables')
edge_mode = W.Dropdown(options=['auto', 'cooccurrence', 'paper_chain'], value='auto', description='Edge mode')
link_prediction_backend = W.Dropdown(options=['auto', 'heuristic', 'pygt_temporal', 'tgn'], value='auto', description='Link pred')
annoy_n_trees = W.IntSlider(value=32, min=4, max=64, step=4, description='Annoy trees', continuous_update=False)
annoy_top_k = W.IntSlider(value=6, min=1, max=20, step=1, description='Annoy top-k', continuous_update=False)

llm_route = W.Dropdown(
    options=[
        ('System default', 'default'),
        ('g4f', 'g4f'),
        ('Local / Ollama', 'local'),
        ('Explicit provider/model', 'explicit'),
    ],
    value='g4f',
    description='LLM route',
)
try:
    _g4f_options = ['auto'] + [m for m in list_available_g4f_models(include_auto=False) if m]
except Exception:
    _g4f_options = ['auto', 'gpt-4o-mini', 'deepseek-r1']
if TASK3_DEFAULT_G4F_MODEL not in _g4f_options:
    _g4f_options.insert(1, TASK3_DEFAULT_G4F_MODEL)
g4f_model = W.Dropdown(options=_g4f_options, value=TASK3_DEFAULT_G4F_MODEL if TASK3_DEFAULT_G4F_MODEL in _g4f_options else 'auto', description='g4f model', layout=W.Layout(width='100%'))
local_text_model = W.Text(value='', description='Local text', placeholder='например, llama3.1:8b', layout=W.Layout(width='100%'))
explicit_llm_provider = W.Text(value='', description='LLM provider', placeholder='openai / mock / anthropic ...', layout=W.Layout(width='100%'))
explicit_llm_model = W.Text(value='', description='LLM model', placeholder='model id', layout=W.Layout(width='100%'))

vlm_backend = W.Dropdown(
    options=[
        ('auto', 'auto'),
        ('none', 'none'),
        ('g4f', 'g4f'),
        ('qwen2_vl (HF local)', 'qwen2_vl'),
        ('qwen3_vl (HF local)', 'qwen3_vl'),
        ('llava (HF local)', 'llava'),
        ('phi3_vision (HF local)', 'phi3_vision'),
    ],
    value='qwen2_vl',
    description='VLM backend',
)
vlm_model_id = W.Text(value=TASK3_DEFAULT_LOCAL_VLM_MODEL, description='VLM model id', placeholder='Qwen/Qwen2.5-VL-3B-Instruct', layout=W.Layout(width='100%'))

create_offline_form = W.Checkbox(value=True, description='Build offline A/B HTML')
create_expert_bundle = W.Checkbox(value=True, description='Build expert artifacts ZIP')
auto_download_offline = W.Checkbox(value=False, description='Auto-download offline HTML (Colab)')
auto_download_bundle = W.Checkbox(value=False, description='Auto-download expert ZIP (Colab)')

status = W.HTML()
summary_html = W.HTML()
out = W.Output(layout=W.Layout(border='1px solid var(--jp-border-color2, #e2e8f0)', padding='8px'))

last_paths_html = W.HTML('<div class="task3-note"><b>Артефакты ещё не созданы.</b></div>')
download_offline_btn = W.Button(description='Скачать offline HTML', button_style='primary')
download_offline_btn.disabled = True
download_bundle_btn = W.Button(description='Скачать expert ZIP', button_style='warning')
download_bundle_btn.disabled = True
open_md_btn = W.Button(description='Показать hypotheses.md', button_style='info')
open_md_btn.disabled = True
artifact_out = W.Output()


def _set_task3_paths(paths):
    global TASK3_LAST_ARTIFACTS
    TASK3_LAST_ARTIFACTS = paths
    offline_p = paths.get('offline_form') if isinstance(paths, dict) else None
    bundle_p = paths.get('expert_bundle') if isinstance(paths, dict) else None
    md_p = paths.get('hypotheses_md') if isinstance(paths, dict) else None
    download_offline_btn.disabled = not (offline_p and Path(offline_p).exists())
    download_bundle_btn.disabled = not (bundle_p and Path(bundle_p).exists())
    open_md_btn.disabled = not (md_p and Path(md_p).exists())
    parts = []
    if paths:
        for key in ['bundle_dir', 'manifest', 'hypotheses_json', 'hypotheses_md', 'offline_form', 'expert_bundle']:
            if paths.get(key):
                parts.append(f'<div><b>{key}</b>: <code>{_escape_html(str(paths[key]))}</code></div>')
    last_paths_html.value = '<div class="task3-note">' + ''.join(parts) + '</div>' if parts else '<div class="task3-note">Нет артефактов.</div>'


def _download_offline(_):
    artifact_out.clear_output()
    paths = TASK3_LAST_ARTIFACTS or {}
    target = Path(paths.get('offline_form')) if paths.get('offline_form') else None
    if target is None or not target.exists():
        status.value = '<div><b>Offline HTML пока не готов.</b></div>'
        return
    _download_file_if_possible(target, status_widget=status)


def _download_bundle(_):
    artifact_out.clear_output()
    paths = TASK3_LAST_ARTIFACTS or {}
    target = Path(paths.get('expert_bundle')) if paths.get('expert_bundle') else None
    if target is None or not target.exists():
        status.value = '<div><b>Expert ZIP пока не готов.</b></div>'
        return
    _download_file_if_possible(target, status_widget=status)


def _show_md(_):
    artifact_out.clear_output()
    paths = TASK3_LAST_ARTIFACTS or {}
    target = Path(paths.get('hypotheses_md')) if paths.get('hypotheses_md') else None
    if target is None or not target.exists():
        status.value = '<div><b>Markdown с гипотезами пока не готов.</b></div>'
        return
    with artifact_out:
        clear_output()
        print(target.read_text(encoding='utf-8'))


download_offline_btn.on_click(_download_offline)
download_bundle_btn.on_click(_download_bundle)
open_md_btn.on_click(_show_md)

left = W.VBox([
    W.HTML('<h3>Входные данные</h3>'),
    input_mode,
    trajectory_upload,
    processed_upload,
    query,
    identifiers_text,
    commands_text,
    commands_upload,
])
right = W.VBox([
    W.HTML('<h3>Эксперт и параметры</h3>'),
    expert_last,
    expert_first,
    expert_pat,
    domain_id,
    out_dir,
    search_limit,
    top_papers,
    top_hypotheses,
    candidate_top_k,
    include_multimodal,
    run_vlm,
    edge_mode,
    link_prediction_backend,
    annoy_n_trees,
    annoy_top_k,
])
models_box = W.VBox([
    W.HTML('<h3>Routing моделей</h3>'),
    llm_route,
    g4f_model,
    local_text_model,
    explicit_llm_provider,
    explicit_llm_model,
    vlm_backend,
    vlm_model_id,
    create_offline_form,
    create_expert_bundle,
    auto_download_offline,
    auto_download_bundle,
])

ui = W.VBox([
    W.HBox([left, right, models_box], layout=W.Layout(align_items='flex-start')),
    status,
    summary_html,
    W.HBox([download_offline_btn, download_bundle_btn, open_md_btn]),
    last_paths_html,
    artifact_out,
    out,
])

# Headless smoke defaults for automated validation
if os.environ.get('TASK3_NOTEBOOK_SMOKE') == '1':
    _smoke_query = (os.environ.get('TASK3_NOTEBOOK_SMOKE_QUERY') or 'temporal catalyst latency forecasting').strip()
    _smoke_processed = (os.environ.get('TASK3_NOTEBOOK_SMOKE_PROCESSED_DIR') or '').strip()
    _smoke_out_dir = (os.environ.get('TASK3_NOTEBOOK_SMOKE_OUT_DIR') or 'runs/task3_notebook_smoke').strip()
    _smoke_vlm_backend = (os.environ.get('TASK3_NOTEBOOK_SMOKE_VLM_BACKEND') or 'none').strip() or 'none'
    _smoke_vlm_model_id = (os.environ.get('TASK3_NOTEBOOK_SMOKE_VLM_MODEL_ID') or 'mock-local-vlm').strip() or 'mock-local-vlm'

    input_mode.value = 'commands'
    query.value = _smoke_query
    commands_text.value = (
        f"query: {_smoke_query}\n"
        + (f"processed_dir: {_smoke_processed}\n" if _smoke_processed else "")
        + f"out_dir: {_smoke_out_dir}\n"
        + "top_papers: 0\n"
        + "top_hypotheses: 3\n"
        + "candidate_top_k: 5\n"
        + "include_multimodal: true\n"
        + "run_vlm: false\n"
        + "edge_mode: cooccurrence\n"
        + "link_prediction_backend: heuristic\n"
        + "llm_provider: mock\n"
        + "llm_model: mock\n"
        + f"vlm_backend: {_smoke_vlm_backend}\n"
        + f"vlm_model_id: {_smoke_vlm_model_id}\n"
    )
    expert_last.value = os.environ.get('TASK3_NOTEBOOK_SMOKE_EXPERT_LAST', 'Smoke')
    expert_first.value = os.environ.get('TASK3_NOTEBOOK_SMOKE_EXPERT_FIRST', 'Validation')
    expert_pat.value = os.environ.get('TASK3_NOTEBOOK_SMOKE_EXPERT_PAT', '-')
    domain_id.value = os.environ.get('TASK3_NOTEBOOK_SMOKE_DOMAIN_ID', 'science')
    out_dir.value = _smoke_out_dir
    search_limit.value = 5
    top_papers.value = 0
    top_hypotheses.value = 3
    candidate_top_k.value = 5
    include_multimodal.value = True
    run_vlm.value = False
    edge_mode.value = 'cooccurrence'
    link_prediction_backend.value = 'heuristic'
    llm_route.value = 'explicit'
    explicit_llm_provider.value = 'mock'
    explicit_llm_model.value = 'mock'
    local_text_model.value = ''
    vlm_backend.value = _smoke_vlm_backend if _smoke_vlm_backend in {value for _, value in vlm_backend.options} else 'none'
    vlm_model_id.value = _smoke_vlm_model_id
    create_offline_form.value = True
    create_expert_bundle.value = True
    auto_download_offline.value = False
    auto_download_bundle.value = False
    status.value = '<div><b>Smoke mode.</b> Notebook prefilled from TASK3_NOTEBOOK_SMOKE_* env vars.</div>'

display(ui)

In [ ]:
# @title
# Запуск Task 3 из отдельной ячейки
# Важно: после заполнения формы выше запускайте именно ЭТУ ячейку.


def run_task3_from_form():
    global TASK3_LAST_RUN, TASK3_LAST_TASK_META
    with out:
        clear_output()
        try:
            status.value = '<div><b>Task 3: подготовка запуска...</b></div>'
            workdir = Path(tempfile.mkdtemp(prefix='task3_notebook_'))

            trajectory_path = _materialize_upload(trajectory_upload.value, workdir, 'task1.yaml')
            commands_file_path = _materialize_upload(commands_upload.value, workdir, 'commands.txt')
            processed_dir = _extract_processed_zip(processed_upload.value, workdir)

            commands = _merged_commands_from_widgets(commands_text.value, commands_upload.value)
            trajectory_doc = _load_yaml_if_present(trajectory_path) if trajectory_path else {}
            trajectory_for_run = trajectory_path if input_mode.value == 'yaml' else None
            task_meta = _task3_task_meta_from_widgets(trajectory_doc if trajectory_for_run else {})
            TASK3_LAST_TASK_META = task_meta

            inline_identifiers = _parse_identifier_blob(identifiers_text.value)
            identifiers = list(inline_identifiers)
            identifiers.extend(commands.get('identifiers') or [])
            deduped = []
            seen = set()
            for item in identifiers:
                item = str(item).strip()
                if not item or item in seen:
                    continue
                seen.add(item)
                deduped.append(item)
            identifiers = deduped

            effective_query = str(query.value or '').strip()
            if not effective_query:
                effective_query = str(commands.get('query') or '').strip()
            if not effective_query and trajectory_doc:
                effective_query = str(trajectory_doc.get('topic') or '').strip()

            if input_mode.value == 'yaml' and trajectory_path is None:
                raise ValueError('Выбран режим Task 1 YAML, но YAML-файл не загружен.')
            if input_mode.value == 'commands' and not (commands_text.value.strip() or commands_file_path):
                raise ValueError('Выбран режим Commands, но commands text/file пустой.')
            if input_mode.value == 'query' and not effective_query and not identifiers:
                raise ValueError('Введите query и/или identifiers.')

            if processed_dir is None and commands.get('processed_dir'):
                candidate = Path(str(commands.get('processed_dir'))).expanduser()
                if candidate.exists():
                    processed_dir = candidate

            llm_kwargs = {'llm_provider': None, 'llm_model': None, 'g4f_model': None, 'local_model': None}
            if llm_route.value == 'g4f':
                llm_kwargs['g4f_model'] = str(g4f_model.value or 'auto').strip() or 'auto'
            elif llm_route.value == 'local':
                llm_kwargs['local_model'] = str(local_text_model.value or '').strip() or None
            elif llm_route.value == 'explicit':
                llm_kwargs['llm_provider'] = str(explicit_llm_provider.value or '').strip() or None
                llm_kwargs['llm_model'] = str(explicit_llm_model.value or '').strip() or None

            if commands.get('g4f_model') and llm_kwargs['g4f_model'] is None and llm_route.value in {'default', 'g4f'}:
                llm_kwargs['g4f_model'] = str(commands['g4f_model']).strip()
            if commands.get('local_model') and llm_kwargs['local_model'] is None and llm_route.value in {'default', 'local'}:
                llm_kwargs['local_model'] = str(commands['local_model']).strip()
            if commands.get('llm_provider') and llm_kwargs['llm_provider'] is None:
                llm_kwargs['llm_provider'] = str(commands['llm_provider']).strip()
            if commands.get('llm_model') and llm_kwargs['llm_model'] is None:
                llm_kwargs['llm_model'] = str(commands['llm_model']).strip()

            selected_vlm_backend = str(commands.get('vlm_backend') or vlm_backend.value or 'auto').strip()
            selected_vlm_model_id = str(commands.get('vlm_model_id') or vlm_model_id.value or '').strip() or None
            selected_domain_id = str(commands.get('domain_id') or domain_id.value or 'science').strip() or 'science'
            selected_out_dir = Path(str(commands.get('out_dir') or out_dir.value or 'runs/task3_hypotheses')).expanduser()

            include_mm_value = include_multimodal.value if commands.get('include_multimodal') is None else bool(commands['include_multimodal'])
            run_vlm_value = run_vlm.value if commands.get('run_vlm') is None else bool(commands['run_vlm'])
            if selected_vlm_backend == 'none':
                run_vlm_value = False

            display(Markdown(f"""
# Запуск Task 3
- **input_mode**: `{input_mode.value}`
- **effective_query**: `{effective_query or '—'}`
- **trajectory**: `{str(trajectory_for_run) if trajectory_for_run else '—'}`
- **identifiers**: `{len(identifiers)}`
- **processed_dir**: `{str(processed_dir) if processed_dir else '—'}`
- **domain_id**: `{selected_domain_id}`
- **LLM route**: `{llm_route.value}`
- **g4f_model**: `{llm_kwargs['g4f_model'] or '—'}`
- **local_model**: `{llm_kwargs['local_model'] or '—'}`
- **vlm_backend**: `{selected_vlm_backend}`
- **vlm_model_id**: `{selected_vlm_model_id or '—'}`
"""))

            bundle = prepare_task3_hypothesis_bundle(
                trajectory=trajectory_for_run,
                query=effective_query,
                identifiers=identifiers,
                processed_dir=processed_dir,
                out_dir=selected_out_dir,
                domain_id=selected_domain_id,
                search_limit=int(commands.get('search_limit') or search_limit.value),
                top_papers=int(commands.get('top_papers') or top_papers.value),
                top_hypotheses=int(commands.get('top_hypotheses') or top_hypotheses.value),
                candidate_top_k=int(commands.get('candidate_top_k') or candidate_top_k.value),
                include_multimodal=bool(include_mm_value),
                run_vlm=bool(run_vlm_value),
                edge_mode=str(commands.get('edge_mode') or edge_mode.value),
                link_prediction_backend=str(commands.get('link_prediction_backend') or link_prediction_backend.value),
                annoy_n_trees=int(annoy_n_trees.value),
                annoy_top_k=int(annoy_top_k.value),
                llm_provider=llm_kwargs['llm_provider'],
                llm_model=llm_kwargs['llm_model'],
                g4f_model=llm_kwargs['g4f_model'],
                local_model=llm_kwargs['local_model'],
                vlm_backend=selected_vlm_backend,
                vlm_model_id=selected_vlm_model_id,
            )
            TASK3_LAST_RUN = bundle

            manifest = json.loads(Path(bundle.manifest_path).read_text(encoding='utf-8'))
            offline_form_path = None
            expert_bundle_path = None
            hypotheses_md = Path(bundle.bundle_dir) / 'hypotheses_ranked.md'

            if bool(create_offline_form.value):
                offline_form_path = build_task3_offline_review_package(manifest, task_meta)
            if bool(create_expert_bundle.value):
                expert_bundle_path = build_task3_expert_artifact_bundle(manifest, task_meta)

            _set_task3_paths({
                'bundle_dir': bundle.bundle_dir,
                'manifest': bundle.manifest_path,
                'hypotheses_json': bundle.hypotheses_path,
                'hypotheses_md': hypotheses_md,
                'offline_form': offline_form_path,
                'expert_bundle': expert_bundle_path,
            })

            top_summary = _top_hypotheses_summary(Path(bundle.hypotheses_path))
            status.value = (
                '<div><b>Task 3 завершён.</b><br>'
                f'Bundle: <code>{_escape_html(str(bundle.bundle_dir))}</code><br>'
                f'Manifest: <code>{_escape_html(str(bundle.manifest_path))}</code><br>'
                f'Hypotheses: <code>{_escape_html(str(bundle.hypotheses_path))}</code>'
                '</div>'
            )
            summary_html.value = (
                '<div>'
                f'<b>Topic:</b> {_escape_html(task_meta.get("topic") or manifest.get("query") or "—")}<br>'
                f'<b>Submission:</b> <code>{_escape_html(task_meta.get("submission_id") or "—")}</code><br>'
                f'<b>Offline HTML:</b> <code>{_escape_html(str(offline_form_path) if offline_form_path else "—")}</code><br>'
                f'<b>Expert ZIP:</b> <code>{_escape_html(str(expert_bundle_path) if expert_bundle_path else "—")}</code><br>'
                f'<hr><b>Top hypotheses</b><br>{top_summary.replace(chr(10), "<br>")}'
                '</div>'
            )

            display(Markdown('## Top hypotheses'))
            display(Markdown(top_summary))

            if auto_download_offline.value and offline_form_path:
                _download_file_if_possible(Path(offline_form_path), status_widget=status)
            if auto_download_bundle.value and expert_bundle_path:
                _download_file_if_possible(Path(expert_bundle_path), status_widget=status)

            return bundle
        except Exception as e:
            status.value = f'<div><b>Ошибка запуска Task 3.</b><br><code>{_escape_html(type(e).__name__ + ": " + str(e))}</code></div>'
            raise
        finally:
            gc.collect()


TASK3_LAST_BUNDLE = run_task3_from_form()

# CLI-режим (тот же workflow без notebook)

```bash
pip install -e ".[task3]"

top-papers-graph task3-bundle   --query "temporal knowledge graph multimodal hypothesis generation"   --top-papers 12   --top-hypotheses 8   --g4f-model auto   --vlm-backend qwen2_vl   --vlm-model-id Qwen/Qwen2.5-VL-3B-Instruct
```

Офлайн запуск по уже готовым `processed_papers`:

```bash
top-papers-graph task3-bundle   --processed-dir runs/some_previous_run/processed_papers   --query "offline smoke"   --edge-mode cooccurrence   --link-prediction-backend heuristic   --no-vlm
```

# FAQ

## Что выбрать для локальной VL модели?
Для локального VLM этот блокнот рассчитан на модели, которые можно загрузить через `Transformers`, например семейство **Qwen2.5-VL / Qwen3-VL**, что согласуется с текущей документацией Transformers для image-text workflows. citeturn778776search0turn778776search1

## Чем отличается offline A/B форма от expert ZIP?
- **offline A/B HTML** — автономная страница для эксперта, где он сравнивает две версии гипотезы и выгружает review ZIP.
- **expert artifacts ZIP** — пакет evidence и служебных артефактов Task 3, который можно передать эксперту вместе с HTML.

## Можно ли запустить Task 3 только по DOI/URL без query?
Да. Достаточно заполнить поле `IDs` или передать `identifier:` строки в commands.

## Можно ли использовать уже подготовленные артефакты Task 2 / ingest?
Да. Загрузите `processed_papers.zip` или укажите `processed_dir` в commands. Тогда Task 3 сможет отработать в более офлайн-режиме, не скачивая статьи заново.